In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import random
%matplotlib inline

In [5]:
import zipfile

In [8]:
path = os.path.abspath(os.path.join(os.getcwd(), '..'))
path = os.path.join(path, 'data_ws2025.zip')
print(path)

/Users/hualesan/anaconda_projects/data_ws2025.zip


In [16]:
dfs = []

with zipfile.ZipFile(path) as z:
    print("Files in zip:", z.namelist())

    csv_files = [
        f for f in z.namelist()
        if f.endswith(".csv") and not f.startswith("__MACOSX/")
    ]

    for file in csv_files:
        ticker = os.path.splitext(os.path.basename(file))[0]

        with z.open(file) as f:
            df = pd.read_csv(f)
            df["Date"] = pd.to_datetime(df["Date"])
            df["Ticker"] = ticker
            dfs.append(df)

stocks = pd.concat(dfs, ignore_index=True)
stocks.head()

Files in zip: ['data_ws2025/', '__MACOSX/._data_ws2025', 'data_ws2025/UNP.csv', '__MACOSX/data_ws2025/._UNP.csv', 'data_ws2025/DHI.csv', '__MACOSX/data_ws2025/._DHI.csv', 'data_ws2025/KEY.csv', '__MACOSX/data_ws2025/._KEY.csv', 'data_ws2025/BMY.csv', '__MACOSX/data_ws2025/._BMY.csv']


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker
0,2021-01-04,187.101,187.991,181.111,182.577,2642400,0.0,0.0,UNP
1,2021-01-05,182.478,185.662,182.442,184.817,2127300,0.0,0.0,UNP
2,2021-01-06,184.862,190.312,184.484,188.684,2512400,0.0,0.0,UNP
3,2021-01-07,189.412,191.876,188.072,191.310,2023000,0.0,0.0,UNP
4,2021-01-08,196.058,199.008,195.150,196.796,3922900,0.0,0.0,UNP


In [18]:
stocks_ts = stocks.set_index("Date").sort_index()
stocks_ts.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker
Date,,,,,,,,
2021-01-04,187.101,187.991,181.111,182.577,2642400,0.0,0.0,UNP
2021-01-04,66.085,66.333,62.897,64.272,3439300,0.0,0.0,DHI
2021-01-04,51.226,51.566,50.197,50.927,12429600,0.0,0.0,BMY
2021-01-04,13.182,13.182,12.745,12.952,8488000,0.0,0.0,KEY
2021-01-05,50.703,51.201,50.172,51.068,11584600,0.0,0.0,BMY


In [19]:
same_high_low = stocks[stocks["High"] == stocks["Low"]].copy()
same_high_low[["Ticker", "Date", "High", "Low"]].head(20)

,Ticker,Date,High,Low


In [20]:
same_high_low["Ticker"].unique()

array([], dtype=object)

In [21]:
same_high_low.groupby("Ticker").size().sort_values(ascending=False)

Series([], dtype: int64)

In [26]:
price_extremes = stocks.groupby("Ticker").agg(
    highest_price=("High", "max"),
    lowest_price=("Low", "min")
)

price_extremes

,highest_price,lowest_price
Ticker,,
BMY,71.308,36.776
DHI,197.639,57.378
KEY,22.364,7.392
UNP,256.930,171.162


In [33]:
stocks["log_return"] = stocks.groupby("Ticker")["Close"].transform(
    lambda s: np.log(s / s.shift(1))
)

In [34]:
stocks = stocks.sort_values(["Ticker", "Date"]).copy()

stocks["log_return"] = stocks.groupby("Ticker")["Close"].transform(
    lambda s: np.log(s / s.shift(1))
)

stocks.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker,log_return
3012,2021-01-04,51.226,51.566,50.197,50.927,12429600,0.0,0.0,BMY,NaN
3013,2021-01-05,50.703,51.201,50.172,51.068,11584600,0.0,0.0,BMY,0.002765
3014,2021-01-06,50.238,51.483,50.213,51.193,12426200,0.0,0.0,BMY,0.002445
3015,2021-01-07,50.894,51.964,50.612,51.740,10931900,0.0,0.0,BMY,0.010628
3016,2021-01-08,51.599,52.420,51.566,51.848,10010500,0.0,0.0,BMY,0.002085


In [29]:
log_return_stats = stocks.groupby("Ticker")["log_return"].agg(
    min="min",
    max="max",
    mean="mean",
    std="std",
    skew="skew"
)

log_return_stats

,min,max,mean,std,skew
Ticker,,,,,
BMY,-0.088998,0.108336,0.000047,0.013780,0.434947
DHI,-0.096940,0.106525,0.000764,0.021989,-0.133059
KEY,-0.319284,0.145886,0.000242,0.025838,-1.509814
UNP,-0.070459,0.099077,0.000203,0.014168,0.481055


In [30]:
def second_highest_gain_date(group):
    g = group.dropna(subset=["log_return"]).sort_values("log_return", ascending=False)
    if len(g) >= 2:
        return g.iloc[1]["Date"]
    return pd.NaT

def second_highest_loss_date(group):
    g = group.dropna(subset=["log_return"]).sort_values("log_return", ascending=True)
    if len(g) >= 2:
        return g.iloc[1]["Date"]
    return pd.NaT

second_dates = stocks.groupby("Ticker").apply(
    lambda g: pd.Series({
        "second_highest_gain_date": second_highest_gain_date(g),
        "second_highest_loss_date": second_highest_loss_date(g)
    })
)

second_dates

/var/folders/yf/34hlxb0s4_5797j67jnwjmhh0000gn/T/ipykernel_1331/2220464078.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  second_dates = stocks.groupby("Ticker").apply(


,second_highest_gain_date,second_highest_loss_date
Ticker,,
BMY,2024-11-11,2023-10-26
DHI,2024-07-18,2021-05-12
KEY,2023-05-05,2023-05-02
UNP,2023-02-27,2022-04-01


In [35]:
weekly_volume = (
    stocks
    .set_index("Date")
    .groupby("Ticker")["Volume"]
    .resample("W")
    .sum()
)

weekly_volume.head(10)

Ticker  Date      
BMY     2021-01-10    57382800
        2021-01-17    68011900
        2021-01-24    43593600
        2021-01-31    67909500
        2021-02-07    74661800
        2021-02-14    63911800
        2021-02-21    55133600
        2021-02-28    70483100
        2021-03-07    62796100
        2021-03-14    51649500
Name: Volume, dtype: int64

In [36]:
median_weekly_volume = weekly_volume.groupby("Ticker").median().to_frame("median_weekly_volume")
median_weekly_volume

,median_weekly_volume
Ticker,
BMY,53123000.0
DHI,13325300.0
KEY,51918200.0
UNP,12371400.0


In [39]:
total_log_return = stocks.groupby("Ticker")["log_return"].sum().sort_values()

total_log_return


Ticker
BMY    0.047511
UNP    0.203505
KEY    0.242413
DHI    0.766641
Name: log_return, dtype: float64

In [40]:
worst_company = total_log_return.idxmin()
worst_value = total_log_return.min()

worst_company, worst_value

('BMY', 0.047511139758637896)

In [41]:
print("Company with the lowest total logarithmic return:", worst_company)
print("Total logarithmic return:", worst_value)

Company with the lowest total logarithmic return: BMY
Total logarithmic return: 0.047511139758637896
